# Check 3: Can we train a flood model on Sen1Floods11?

**Goal of this run:** make sure the full pipeline works end to end. Accuracy does not matter yet.

**What this notebook does**
1. Downloads only the hand-labelled part of Sen1Floods11 (446 image chips, not the full 14 GB).
2. Shows a few examples so we understand the data.
3. Builds the simple threshold baseline ("dark pixels are water").
4. Trains one small U-Net for a few epochs.
5. Compares the two on the validation set.

**Before you start:** in Colab, go to *Runtime > Change runtime type* and choose **T4 GPU**.

We never touch the **test** set here. It is kept for the final evaluation only.

In [ ]:
!pip install -q segmentation-models-pytorch rasterio

In [ ]:
# Get the project code (config, preprocessing, metrics) from GitHub.
# Change REPO_URL if your repository lives somewhere else.
import os, sys, subprocess
REPO_URL = "https://github.com/Matrixco123/cutoff-flood.git"
REPO = "/content/cutoff-flood"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from src import config
print("Project code loaded from", REPO)

In [ ]:
# Download the hand-labelled data (about 1 to 2 GB).
# If gsutil asks for login, run:  from google.colab import auth; auth.authenticate_user()
DATA = "/content/sen1floods11"
BUCKET = "gs://sen1floods11/v1.1"
!mkdir -p {DATA}
!gsutil -m -q cp -r {BUCKET}/data/flood_events/HandLabeled/S1Hand {DATA}/
!gsutil -m -q cp -r {BUCKET}/data/flood_events/HandLabeled/LabelHand {DATA}/
!gsutil -m -q cp -r {BUCKET}/splits/flood_handlabeled {DATA}/
!ls {DATA}; ls {DATA}/flood_handlabeled
!echo "S1 chips: $(ls {DATA}/S1Hand | wc -l)   labels: $(ls {DATA}/LabelHand | wc -l)"

In [ ]:
import glob, re
import numpy as np
import pandas as pd
import rasterio
from src.preprocess import preprocess_sar, clean_label
from src.metrics import Counts

def chip_ids(csv_path):
    """Read a split file and return chip ids like 'Ghana_103272'."""
    df = pd.read_csv(csv_path, header=None)
    names = df.iloc[:, 0].astype(str)
    return [re.sub(r"_(S1Hand|LabelHand|S2Hand|S1Weak).*$", "", os.path.basename(n)) for n in names]

def find_split(word):
    files = [f for f in glob.glob(f"{DATA}/flood_handlabeled/*.csv")
             if word in os.path.basename(f).lower() and "bolivia" not in f.lower()]
    assert files, f"No split file containing '{word}' found"
    return files[0]

train_ids = chip_ids(find_split("train"))
valid_ids = chip_ids(find_split("valid"))
print(f"train: {len(train_ids)} chips   valid: {len(valid_ids)} chips")

def load_chip(cid):
    with rasterio.open(f"{DATA}/S1Hand/{cid}_S1Hand.tif") as f:
        img = f.read()          # (2, 512, 512): VV, VH in dB
    with rasterio.open(f"{DATA}/LabelHand/{cid}_LabelHand.tif") as f:
        lab = f.read()
    return img, clean_label(lab)

raw0, lab0 = load_chip(train_ids[0])
print("image shape", raw0.shape, "| VV range (dB)", np.nanmin(raw0[0]).round(1), "to", np.nanmax(raw0[0]).round(1))
print("label values", np.unique(lab0))

In [ ]:
# Look at a few examples: VV, VH, and the hand label (white = water, grey = no data)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 3, figsize=(10, 13))
for row, cid in zip(axes, train_ids[:: max(1, len(train_ids) // 4)][:4]):
    raw, lab = load_chip(cid)
    x = preprocess_sar(raw)
    row[0].imshow(x[0], cmap="gray"); row[0].set_title(f"{cid}  VV")
    row[1].imshow(x[1], cmap="gray"); row[1].set_title("VH")
    show = np.where(lab == -1, 0.5, lab.astype(float))
    row[2].imshow(show, cmap="gray", vmin=0, vmax=1); row[2].set_title("Label")
    for a in row: a.axis("off")
plt.tight_layout(); plt.show()

## Baseline: a simple threshold

Pick one VV threshold from the training data using Otsu's method, then call every pixel darker than it "water". This is the method our U-Net has to beat.

In [ ]:
from skimage.filters import threshold_otsu

# Sample training pixels to choose the threshold
rng = np.random.default_rng(0)
samples = []
for cid in train_ids:
    raw, lab = load_chip(cid)
    vv = preprocess_sar(raw)[0][lab != -1]
    if vv.size:
        samples.append(rng.choice(vv, size=min(5000, vv.size), replace=False))
thr = threshold_otsu(np.concatenate(samples))
print("Otsu threshold on scaled VV:", round(float(thr), 3))

base = Counts()
for cid in valid_ids:
    raw, lab = load_chip(cid)
    base.update(preprocess_sar(raw)[0] < thr, lab)
print("Threshold baseline on validation:", base.summary())

## U-Net: dataset and model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)
torch.manual_seed(config.SEEDS[0]); np.random.seed(config.SEEDS[0])

class FloodChips(Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = ids, augment
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        raw, lab = load_chip(self.ids[i])
        x = preprocess_sar(raw)
        if self.augment:   # random flips and 90-degree turns
            if np.random.rand() < 0.5: x, lab = x[:, :, ::-1], lab[:, ::-1]
            if np.random.rand() < 0.5: x, lab = x[:, ::-1, :], lab[::-1, :]
            k = np.random.randint(4); x, lab = np.rot90(x, k, (1, 2)), np.rot90(lab, k)
        return torch.from_numpy(x.copy()), torch.from_numpy(lab.copy())

train_dl = DataLoader(FloodChips(train_ids, augment=True), batch_size=8, shuffle=True, num_workers=2)
valid_dl = DataLoader(FloodChips(valid_ids), batch_size=8, num_workers=2)

model = smp.Unet(config.ENCODER, encoder_weights="imagenet", in_channels=2, classes=1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

def masked_bce(logits, lab):
    """Binary cross-entropy that ignores no-data pixels (label -1)."""
    valid = lab != -1
    target = lab.clamp(min=0).float()
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits[:, 0], target, reduction="none")
    return (loss * valid).sum() / valid.sum().clamp(min=1)

def evaluate(model, dl):
    model.eval(); c = Counts()
    with torch.no_grad():
        for x, lab in dl:
            pred = (torch.sigmoid(model(x.to(device)))[:, 0] > 0.5).cpu().numpy()
            c.update(pred, lab.numpy())
    return c

In [ ]:
EPOCHS = 10   # small run; the real training will be longer
history = []
for epoch in range(1, EPOCHS + 1):
    model.train(); total = 0
    for x, lab in train_dl:
        x, lab = x.to(device), lab.to(device)
        loss = masked_bce(model(x), lab)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(x)
    val = evaluate(model, valid_dl).summary()
    history.append({"epoch": epoch, "train_loss": total / len(train_dl.dataset), **val})
    print(f"epoch {epoch:2d}  loss {history[-1]['train_loss']:.4f}  val IoU {val['IoU']:.3f}")

torch.save(model.state_dict(), "/content/unet_first_run.pth")

In [ ]:
# Compare and look at predictions
hist = pd.DataFrame(history)
plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1); plt.plot(hist.epoch, hist.train_loss); plt.title("Training loss"); plt.xlabel("epoch")
plt.subplot(1, 2, 2); plt.plot(hist.epoch, hist.IoU); plt.axhline(base.iou(), ls="--", c="gray", label="threshold baseline")
plt.title("Validation IoU"); plt.xlabel("epoch"); plt.legend(); plt.tight_layout(); plt.show()

print("Threshold baseline:", base.summary())
print("U-Net (last epoch):", {k: v for k, v in history[-1].items() if k not in ("epoch", "train_loss")})

model.eval()
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for row, cid in zip(axes, valid_ids[:3]):
    raw, lab = load_chip(cid); x = preprocess_sar(raw)
    with torch.no_grad():
        p = torch.sigmoid(model(torch.from_numpy(x)[None].to(device)))[0, 0].cpu().numpy()
    row[0].imshow(x[0], cmap="gray"); row[0].set_title(f"{cid} VV")
    row[1].imshow(np.where(lab == -1, 0.5, lab), cmap="gray", vmin=0, vmax=1); row[1].set_title("Label")
    row[2].imshow(p, cmap="Blues", vmin=0, vmax=1); row[2].set_title("U-Net water probability")
    for a in row: a.axis("off")
plt.tight_layout(); plt.show()

## Decision

Write the baseline and U-Net validation numbers in `NOTES.md` (under "Check 3").

- **Pass:** training loss goes down and the predicted water roughly matches the labels. It is fine if the U-Net only slightly beats the threshold after 10 epochs.
- **Fail:** errors, or loss does not go down. This is a code problem to debug, **not** a reason to change the study area.

Save `/content/unet_first_run.pth` to Google Drive if you want to keep it (Colab deletes files when the session ends).